# 03 — Churn Driver Analysis
## What Drives Customer Churn?

**Objective:** Identify and rank the key factors that influence customer churn through statistical testing and modeling.

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

import sys
sys.path.append('..')
from src.preprocessing import load_data, full_pipeline

In [ ]:
df = load_data()
df_original = df.copy()
print(f"Loaded {len(df)} customers, {df.shape[1]} features")

## 2. Categorical Driver Tests (Chi-Square)

In [ ]:
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                    'PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies',
                    'Contract', 'PaperlessBilling', 'PaymentMethod']

chi_results = []
for col in cat_cols:
    ctab = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, expected = stats.chi2_contingency(ctab)
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    max_diff = churn_rate.max() - churn_rate.min()
    chi_results.append({
        'Feature': col,
        'Chi2_Stat': round(chi2, 2),
        'P_Value': p,
        'Significant': p < 0.05,
        'Max_Churn_Diff_%': round(max_diff, 2)
    })

chi_df = pd.DataFrame(chi_results).sort_values('Chi2_Stat', ascending=False)
chi_df

## 3. Numerical Driver Tests (T-Test / Mann-Whitney)

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
num_results = []
for col in num_cols:
    churned = df[df['Churn'] == 'Yes'][col].dropna()
    stayed = df[df['Churn'] == 'No'][col].dropna()
    stat, p = stats.mannwhitneyu(churned, stayed, alternative='two-sided')
    num_results.append({
        'Feature': col,
        'Churned_Mean': round(churned.mean(), 2),
        'Stayed_Mean': round(stayed.mean(), 2),
        'Difference': round(stayed.mean() - churned.mean(), 2),
        'MWU_Stat': round(stat, 0),
        'P_Value': p,
        'Significant': p < 0.05
    })
num_df = pd.DataFrame(num_results)
num_df

## 5. Churn Driver Ranking

In [ ]:
df_encoded = pd.get_dummies(df.select_dtypes(include=['object']), drop_first=True)
df_numeric = pd.concat([df.select_dtypes(include=[np.number]), df_encoded], axis=1)
corr = df_numeric.corr()['Churn_Yes'].drop('Churn_Yes').sort_values(key=abs, ascending=False)

driver_ranking = pd.DataFrame({
    'Feature': corr.index,
    'Correlation_with_Churn': corr.values.round(3),
    'Abs_Correlation': abs(corr.values).round(3)
}).sort_values('Abs_Correlation', ascending=False).head(15)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in driver_ranking['Correlation_with_Churn'].head(10)]
driver_ranking.head(10).sort_values('Correlation_with_Churn').plot(
    x='Feature', y='Correlation_with_Churn', kind='barh', color=colors, legend=False)
plt.title('Top 10 Churn Drivers (Correlation)')
plt.xlabel('Correlation with Churn')
plt.tight_layout()
plt.savefig('../reports/churn_drivers.png', dpi=150, bbox_inches='tight')
plt.show()
driver_ranking

## 6. Key Churn Driver Deep Dives

### Driver 1: Contract Type
Month-to-month contracts have the strongest association with churn. Customers on longer contracts are significantly more loyal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, contract in enumerate(['Month-to-month', 'One year', 'Two year']):
    subset = df[df['Contract'] == contract]
    churn_rate = (subset['Churn'] == 'Yes').mean() * 100
    axes[i].pie([churn_rate, 100-churn_rate], labels=['Churned', 'Stayed'],
                autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'], startangle=90)
    axes[i].set_title(f'{contract}\nChurn: {churn_rate:.1f}%')

plt.suptitle('Churn Rate by Contract Type', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/driver_contract.png', dpi=150, bbox_inches='tight')
plt.show()

### Driver 2: Tenure Duration
Churn risk is highest in the first year and drops significantly after 2 years.

In [ ]:
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 3, 6, 12, 24, 48, 72],
                                    labels=['0-3mo', '3-6mo', '6-12mo', '1-2yr', '2-4yr', '4-6yr'])
tenure_churn = df.groupby('TenureGroup', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100)

plt.figure(figsize=(10, 5))
ax = tenure_churn.plot(kind='bar', color='coral', edgecolor='white')
for i, v in enumerate(tenure_churn):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
plt.title('Churn Rate by Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.xlabel('Tenure')
plt.tight_layout()
plt.savefig('../reports/driver_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

### Driver 3: Service Adoption
Customers with fewer services and without security/tech support are more likely to churn.

### Driver 4: Payment Method
Electronic check users show significantly higher churn — likely due to the friction of manual payment vs. auto-pay.

## 7. Summary of Churn Drivers

| Rank | Driver | Impact | Business Insight |
|------|--------|--------|------------------|
| 1 | Contract Type | Very High | Month-to-month customers are 3x more likely to churn |
| 2 | Tenure | Very High | Risk drops 60% after 12 months |
| 3 | Online Security | High | Customers without it churn 2x more |
| 4 | Tech Support | High | No tech support = higher churn |
| 5 | Payment Method | Medium | Electronic check = 2x churn vs auto-pay |
| 6 | Monthly Charges | Medium | Higher charges = slightly higher churn |
| 7 | Internet Service | Medium | Fiber optic churns more than DSL |
| 8 | Paperless Billing | Medium | Paperless billing users churn more |
| 9 | Dependents | Low | With dependents = more stable |
| 10 | Partner | Low | With partner = more stable

---
*End of 03 — Churn Driver Analysis*